In [17]:
import numpy as np
import pandas as pd
import joblib
from google.colab import files                                   #Comment out if you ar runnin locally
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [18]:
uploaded = files.upload()  #Comment out if you ar runnin locally

Saving G7_WA_Fn-UseC_-Telco-Customer-Churn.csv to G7_WA_Fn-UseC_-Telco-Customer-Churn.csv


In [20]:

df = pd.read_csv('G7_WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


Data Cleaning

In [21]:
# Drop ID
df = df.drop('customerID', axis=1)

# Convert TotalCharges
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Fill missing
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].mean())

Encode Target

In [22]:
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})


Feature Engineering

In [23]:
df['AvgCharges'] = df['TotalCharges'] / (df['tenure'] + 1)
df['HighSpender'] = (df['MonthlyCharges'] > 70).astype(int)

One-Hot Encoding

In [24]:
df = pd.get_dummies(df, drop_first=True)

In [25]:
X = df.drop('Churn', axis=1)
y = df['Churn']

# Save columns
columns = X.columns

In [26]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Scaling

In [27]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [29]:
from xgboost import XGBClassifier

Train XGBoost

In [30]:
model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=3,   # handles imbalance
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

model.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [18:54:01] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, ...)

Evaluation

In [31]:
pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

Accuracy: 0.7757274662881476
              precision    recall  f1-score   support

           0       0.90      0.78      0.84      1036
           1       0.56      0.76      0.64       373

    accuracy                           0.78      1409
   macro avg       0.73      0.77      0.74      1409
weighted avg       0.81      0.78      0.79      1409



Feature Importance

In [32]:
import matplotlib.pyplot as plt

importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print(importance.head(10))

                               Feature  Importance
27                   Contract_Two year    0.269930
12         InternetService_Fiber optic    0.196791
14  OnlineSecurity_No internet service    0.110564
26                   Contract_One year    0.077353
13                  InternetService_No    0.043422
25                 StreamingMovies_Yes    0.023236
1                               tenure    0.022419
9                     PhoneService_Yes    0.019810
30      PaymentMethod_Electronic check    0.019422
10      MultipleLines_No phone service    0.015186


Save Model Files

In [33]:
joblib.dump(model, 'churn_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(columns, 'columns.pkl')

['columns.pkl']